Tune hyperparameters for better model performance using automated search techniques like GridSearchCV, RandomizedSearchCV, and Bayesian Optimization while ensuring model generalization through validation strategies.

GridSearchCV for Ridge Regression

In [4]:
from scipy.sparse import load_npz
import numpy as np
import pandas as pd

In [2]:
x_trained_processed = load_npz("x_trained_processed.npz")
x_test_processed = load_npz("x_test_processed.npz")

y_train = np.load("y_train.npy")
y_test = np.load("y_test.npy")

print("Training features:", x_trained_processed.shape)
print("Testing features:", x_test_processed.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (18786, 3045)
Testing features: (4697, 3045)
Training target: (18786,)
Testing target: (4697,)


GridSearchCV

In [3]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [7]:
# Hyperparameter combinations
param_grid = {
    'alpha': [0.01, 0.1, 1, 10, 100],
    'fit_intercept': [True, False]
}

# Grid Search with 5-fold cross-validation
grid_search = GridSearchCV(
    estimator=Ridge(solver='lsqr'),
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1,
    return_train_score=True
)

# Train using existing processed data
grid_search.fit(x_trained_processed, y_train)

best_grid_model = grid_search.best_estimator_
y_pred_grid = best_grid_model.predict(x_test_processed)

print("Best Parameters:", grid_search.best_params_)
print("Best CV RMSE:", round(-grid_search.best_score_, 4))
print("Test MAE:", round(mean_absolute_error(y_test, y_pred_grid), 4))
print("Test RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_grid)), 4))
print("Test R2:", round(r2_score(y_test, y_pred_grid), 4))

# Comparison
grid_res = pd.DataFrame(grid_search.cv_results_)
display(grid_res[
    [
        'param_alpha',
        'param_fit_intercept',
        'mean_train_score',
        'mean_test_score',
        'rank_test_score'
    ]
].assign(
    Train_RMSE=lambda x: -x['mean_train_score'],
    CV_RMSE=lambda x: -x['mean_test_score']
).sort_values('CV_RMSE').round(4))


Best Parameters: {'alpha': 1, 'fit_intercept': False}
Best CV RMSE: 0.7373
Test MAE: 0.5644
Test RMSE: 0.7472
Test R2: 0.5421


,param_alpha,param_fit_intercept,mean_train_score,mean_test_score,rank_test_score,Train_RMSE,CV_RMSE
5,1.00,False,-0.6419,-0.7373,1,0.6419,0.7373
4,1.00,True,-0.6417,-0.7374,2,0.6417,0.7374
7,10.00,False,-0.7207,-0.7551,3,0.7207,0.7551
6,10.00,True,-0.7208,-0.7552,4,0.7208,0.7552
3,0.10,False,-0.6214,-0.7715,5,0.6214,0.7715
2,0.10,True,-0.6211,-0.7721,6,0.6211,0.7721
1,0.01,False,-0.6198,-0.7862,7,0.6198,0.7862
0,0.01,True,-0.6197,-0.7868,8,0.6197,0.7868
9,100.00,False,-0.8508,-0.8588,9,0.8508,0.8588
8,100.00,True,-0.8508,-0.8589,10,0.8508,0.8589


RandomizedSearchCV

In [8]:
from sklearn.model_selection import RandomizedSearchCV
from  scipy.stats import loguniform

In [11]:
# Define parameter distributions
param_distributions = {
    'alpha': loguniform(0.001, 1000),
    'fit_intercept': [True, False]
}

# Initialize RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=Ridge(solver='lsqr'),
    param_distributions=param_distributions,
    n_iter=20,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1,
    random_state=42,
    return_train_score=True
)

random_search.fit(x_trained_processed, y_train)

best_random_model = random_search.best_estimator_

y_pred_random = best_random_model.predict(x_test_processed)

print("Best Parameters:", random_search.best_params_)
print("Best CV RMSE:", round(-random_search.best_score_, 4))
print("Test MAE:", round(mean_absolute_error(y_test, y_pred_random), 4))
print("Test RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_random)), 4))
print("Test R2:", round(r2_score(y_test, y_pred_random), 4))

random_res = pd.DataFrame(random_search.cv_results_)

display(
    random_res[
        [
            'param_alpha',
            'param_fit_intercept',
            'mean_train_score',
            'mean_test_score',
            'rank_test_score'
        ]
    ].assign(
        Train_RMSE=lambda x: -x['mean_train_score'],
        CV_RMSE=lambda x: -x['mean_test_score']
    ).sort_values('CV_RMSE').round(4).reset_index(drop=True)
)

Best Parameters: {'alpha': np.float64(1.408146893930582), 'fit_intercept': True}
Best CV RMSE: 0.7345
Test MAE: 0.5623
Test RMSE: 0.7445
Test R2: 0.5453


,param_alpha,param_fit_intercept,mean_train_score,mean_test_score,rank_test_score,Train_RMSE,CV_RMSE
0,1.4081,True,-0.6494,-0.7345,1,0.6494,0.7345
1,1.2173,False,-0.6460,-0.7356,2,0.6460,0.7356
2,3.9080,True,-0.6807,-0.7366,3,0.6807,0.7366
3,4.6765,True,-0.6875,-0.7389,4,0.6875,0.7389
4,5.1323,False,-0.6912,-0.7402,5,0.6912,0.7402
5,0.6318,True,-0.6336,-0.7433,6,0.6336,0.7433
6,0.5450,True,-0.6317,-0.7455,7,0.6317,0.7455
7,0.4731,True,-0.6298,-0.7480,8,0.6298,0.7480
8,0.3905,True,-0.6280,-0.7511,9,0.6280,0.7511
9,0.1767,True,-0.6227,-0.7643,10,0.6227,0.7643


Bayesian Optimization

In [12]:
%pip install scikit-optimize


   ---------------------------------------- 0/2 [pyaml]
   -------------------- ------------------- 1/2 [scikit-optimize]
   -------------------- ------------------- 1/2 [scikit-optimize]
   -------------------- ------------------- 1/2 [scikit-optimize]
   -------------------- ------------------- 1/2 [scikit-optimize]
   -------------------- ------------------- 1/2 [scikit-optimize]
   -------------------- ------------------- 1/2 [scikit-optimize]
   -------------------- ------------------- 1/2 [scikit-optimize]
   ---------------------------------------- 2/2 [scikit-optimize]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from skopt import BayesSearchCV
from skopt.space import Real, Categorical

# Define the search space
search_spaces = {
    'alpha': Real(
        0.001,
        1000,
        prior='log-uniform'
    ),
    'fit_intercept': Categorical([True, False])
}

# Initialize Bayesian Optimization
bayes_search = BayesSearchCV(
    estimator=Ridge(solver='lsqr'),
    search_spaces=search_spaces,
    n_iter=20,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1,
    random_state=42,
    return_train_score=True
)

bayes_search.fit(x_trained_processed, y_train)

best_bayes_model = bayes_search.best_estimator_

y_pred_bayes = best_bayes_model.predict(x_test_processed)

print("Best Parameters:", bayes_search.best_params_)
print("Best CV RMSE:", round(-bayes_search.best_score_, 4))
print("Test MAE:", round(mean_absolute_error(y_test, y_pred_bayes), 4))
print("Test RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_bayes)), 4))
print("Test R2:", round(r2_score(y_test, y_pred_bayes), 4))

random_res = pd.DataFrame(bayes_search.cv_results_)

display(
    random_res[
        [
            'param_alpha',
            'param_fit_intercept',
            'mean_train_score',
            'mean_test_score',
            'rank_test_score'
        ]
    ].assign(
        Train_RMSE=lambda x: -x['mean_train_score'],
        CV_RMSE=lambda x: -x['mean_test_score']
    ).sort_values('CV_RMSE').round(4).reset_index(drop=True)
)

Best Parameters: OrderedDict({'alpha': 2.1223385671287014, 'fit_intercept': False})
Best CV RMSE: 0.7331
Test MAE: 0.5608
Test RMSE: 0.7428
Test R2: 0.5474


,param_alpha,param_fit_intercept,mean_train_score,mean_test_score,rank_test_score,Train_RMSE,CV_RMSE
0,2.1223,False,-0.6605,-0.7331,1,0.6605,0.7331
1,1.9835,True,-0.6584,-0.7332,2,0.6584,0.7332
2,2.2732,False,-0.6625,-0.7332,3,0.6625,0.7332
3,1.8215,False,-0.6560,-0.7334,4,0.6560,0.7334
4,3.2139,True,-0.6737,-0.7348,5,0.6737,0.7348
5,1.0516,True,-0.6427,-0.7369,6,0.6427,0.7369
6,5.0405,False,-0.6905,-0.7399,7,0.6905,0.7399
7,0.4667,False,-0.6299,-0.7479,8,0.6299,0.7479
8,0.2888,False,-0.6255,-0.7561,9,0.6255,0.7561
9,0.0483,False,-0.6205,-0.7782,10,0.6205,0.7782


In [15]:
# Predictions on training and test data
y_train_pred = best_bayes_model.predict(x_trained_processed)
y_test_pred = best_bayes_model.predict(x_test_processed)

# Generalization metrics
generalization_results = pd.DataFrame({
    "Dataset": ["Training", "Test"],
    "MAE": [
        mean_absolute_error(y_train, y_train_pred),
        mean_absolute_error(y_test, y_test_pred)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_train, y_train_pred)),
        np.sqrt(mean_squared_error(y_test, y_test_pred))
    ],
    "R2": [
        r2_score(y_train, y_train_pred),
        r2_score(y_test, y_test_pred)
    ]
})

display(generalization_results.round(4))

# Compare training and cross-validation RMSE
train_rmse = np.sqrt(
    mean_squared_error(y_train, y_train_pred)
)
cv_rmse = -bayes_search.best_score_

print("Training RMSE:", round(train_rmse, 4))
print("CV RMSE:", round(cv_rmse, 4))
print("Generalization gap:", round(cv_rmse - train_rmse, 4))


,Dataset,MAE,RMSE,R2
0,Training,0.5026,0.6649,0.6422
1,Test,0.5608,0.7428,0.5474


Training RMSE: 0.6649
CV RMSE: 0.7331
Generalization gap: 0.0683


### Task 29: Hyperparameter Optimization

Three automated hyperparameter optimization techniques—GridSearchCV, RandomizedSearchCV, and Bayesian Optimization—were implemented using Ridge Regression. Five-fold cross-validation was used to evaluate model configurations based on RMSE.

GridSearchCV selected alpha = 1.0 and achieved a CV RMSE of 0.7373. RandomizedSearchCV selected alpha ≈ 1.4081 and achieved a CV RMSE of 0.7345. Bayesian Optimization selected alpha ≈ 2.1223 and achieved the lowest CV RMSE of 0.7331.

Bayesian Optimization also achieved the lowest test MAE (0.5608), lowest test RMSE (0.7428), and highest test R² (0.5474). Therefore, among the three evaluated searches, Bayesian Optimization produced the best results for the selected Ridge Regression model.

These results indicate a modest improvement in predictive performance through hyperparameter tuning. The selected configuration should be retained for final evaluation, with conclusions limited to the current train-test split and validation strategy.